In [2]:
import os
from dotenv import load_dotenv
import cohere as co

load_dotenv()

True

In [11]:
client = co.Client(os.getenv("COHERE_API_HERE"))

def get_embedding(sentences):
    return client.embed(
        model="embed-english-v3.0",
        texts=sentences,
        input_type='search_document',
        embedding_types=['float']
    ).embeddings.float

In [12]:
import pandas as pd

so_df = pd.read_csv('test.csv')
print(so_df.head(5))


   Class Index                                              Title  \
0            3                  Fears for T N pension after talks   
1            4  The Race is On: Second Private Team Sets Launc...   
2            4      Ky. Company Wins Grant to Study Peptides (AP)   
3            4      Prediction Unit Helps Forecast Wildfires (AP)   
4            4        Calif. Aims to Limit Farm-Related Smog (AP)   

                                         Description  
0  Unions representing workers at Turner   Newall...  
1  SPACE.com - TORONTO, Canada -- A second\team o...  
2  AP - A company founded by a chemistry research...  
3  AP - It's barely dawn when Mike Fitzpatrick st...  
4  AP - Southern California's smog-fighting agenc...  


In [13]:
so_df = so_df.sample(n=2000, random_state=42).reset_index(drop=True)
so_df = pd.DataFrame({
    "input_text": so_df["Title"] +" "+so_df["Description"]
})

print(so_df.head(5))

                                          input_text
0  Fan v Fan: Manchester City-Tottenham Hotspur T...
1  Paris Tourists Search for Key to 'Da Vinci Cod...
2  Net firms: Don't tax VoIP The Spanish-American...
3  Dependent species risk extinction The global e...
4  EDS Is Charter Member of Siebel BPO Alliance (...


In [14]:
import time

def generate_batches(sentences, batch_size = 96):
    for i in range(0, len(sentences), batch_size):
        yield sentences[i: min(len(sentences), i+batch_size)]

def generate_embeddings(sentences):
    batches = generate_batches(sentences)
    input_text_embeddings = []
    for batch in batches:
        embeddings = get_embedding(batch)
        input_text_embeddings.extend(embeddings)
        time.sleep(2)
    return input_text_embeddings

In [16]:
so_df["input_text_embedding"] = generate_embeddings(list(so_df["input_text"]))


print(so_df.head(5))

                                          input_text  \
0  Fan v Fan: Manchester City-Tottenham Hotspur T...   
1  Paris Tourists Search for Key to 'Da Vinci Cod...   
2  Net firms: Don't tax VoIP The Spanish-American...   
3  Dependent species risk extinction The global e...   
4  EDS Is Charter Member of Siebel BPO Alliance (...   

                                input_text_embedding  
0  [0.052764893, 0.006160736, 0.0066604614, -0.01...  
1  [-0.00033140182, -0.0090408325, 0.040740967, 0...  
2  [0.060516357, -0.03274536, 0.050567627, -0.027...  
3  [-0.011352539, -0.00957489, -0.013114929, 0.04...  
4  [0.004272461, 0.02041626, -0.016693115, -0.025...  


In [23]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_context(query):
    query_embedding = get_embedding([query])[0]
    cosine_similarity_array = cosine_similarity([query_embedding],
                                                so_df["input_text_embedding"].tolist())
    index_doc_cosine = np.argmax(cosine_similarity_array)
    return so_df.input_text[index_doc_cosine]


In [37]:


query = "Summarize the Terrorism news around the world"

context = get_context(query)

In [38]:
prompt = f"""Here is the context: {context}
             Using the relevant information from the context,
             provide an answer to the query: {query}."
             If the context doesn't provide \
             any relevant information, \
             answer with \
             [I couldn't find a good match in the \
             document database for your query]
             nothing else not even the reasoning
             """


In [41]:
response = client.chat(
    model="command-a-03-2025",
    message=prompt,
    temperature=0.3
)

print(response.text)

Britain has increased its anti-terror security spending in response to the September 11 attacks on the United States, recognizing the potential threat of similar or worse terrorist acts occurring in major British cities like London.
